<a href="https://colab.research.google.com/github/saketjainn/cv-project/blob/main/DEFECT_DETECTION_SYSTEM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# COMPLETE WALL DEFECT DETECTION SYSTEM
# Crack + Drywall Classification using Classical CV + ML

# STEP 1 : INSTALL REQUIRED LIBRARIES

!pip install opencv-python matplotlib scikit-image scikit-learn

In [ ]:
# STEP 2 : IMPORT REQUIRED LIBRARIES

import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

from skimage import measure
from skimage.feature import hog

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

In [ ]:
# STEP 3 : MOUNT GOOGLE DRIVE

from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# STEP 4 : DEFINE DATASET PATHS

crack_path = "/content/drive/MyDrive/CV_Project/augmented_cracks"

drywall_path = "/content/drive/MyDrive/CV_Project/augmented_drywall"

In [ ]:
# STEP 5 : LOAD IMAGE FILES

crack_images = [
    f for f in os.listdir(crack_path)
    if f.endswith('.jpg') or f.endswith('.png')
]

drywall_images = [
    f for f in os.listdir(drywall_path)
    if f.endswith('.jpg') or f.endswith('.png')
]

print("Total Crack Images :", len(crack_images))

print("Total Drywall Images :", len(drywall_images))

Total Crack Images : 200
Total Drywall Images : 200


In [ ]:
# STEP 6 : FEATURE EXTRACTION FUNCTION
def extract_features(image_path):

    image = cv2.imread(image_path)

    image = cv2.resize(image, (256,256))

    gray = cv2.cvtColor(
        image,
        cv2.COLOR_BGR2GRAY
    )

    # Gaussian Blur
    blur = cv2.GaussianBlur(
        gray,
        (5,5),
        0
    )

    # CLAHE
    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8,8)
    )

    contrast = clahe.apply(blur)

    # Adaptive Threshold
    thresh = cv2.adaptiveThreshold(
        contrast,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        11,
        2
    )

    # Morphological Closing
    kernel = np.ones((3,3), np.uint8)

    closing = cv2.morphologyEx(
        thresh,
        cv2.MORPH_CLOSE,
        kernel
    )

    # Canny Edge Detection
    canny = cv2.Canny(
        contrast,
        100,
        200
    )

    # ========================================================
    # FEATURE 1 : CRACK AREA
    # ========================================================

    crack_pixels = np.sum(closing == 255)

    total_pixels = (
        closing.shape[0] *
        closing.shape[1]
    )

    crack_area = crack_pixels / total_pixels

    # ========================================================
    # FEATURE 2 : EDGE DENSITY
    # ========================================================

    edge_pixels = np.sum(canny > 0)

    edge_density = edge_pixels / total_pixels

    # ========================================================
    # FEATURE 3 : CONTOUR COUNT
    # ========================================================

    contours, hierarchy = cv2.findContours(
        closing,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    contour_count = len(contours)

    # FEATURE 4 : HOG FEATURES

    hog_features = hog(
        gray,
        pixels_per_cell=(16,16),
        cells_per_block=(2,2),
        feature_vector=True
    )

    # COMBINE FEATURES

    features = np.hstack([
        crack_area,
        edge_density,
        contour_count,
        hog_features
    ])

    return features, image, gray, thresh, canny, closing